# Peer-pricing pipeline: a local, inspectable walkthrough

This walkthrough reads the ignored local harmonized corpus and macro lookup -> selects dated job-budget proxies using predeclared OOT windows -> fits text and metadata on past rows only -> builds 53D coordinates and nearest peers -> calculates the corridor and fee -> checks local research-artifact replay. It reuses the project modules rather than replacing their business logic.

**Research only; not production or mentor-pricing accuracy evidence:** the dated job rows are real marketplace *posted budgets*, not verified mentor payments. Undated profile asking rates are excluded from OOT. Results on this proxy population cannot pass the deferred empirical mentor-pricing gate (#81/#91), promote an artifact or authorize an API. Firebase, Firestore, credentials and network latency measurement are deliberately excluded.

Read [the overview](../README.md), [glossary](../docs/README.md), [provenance audit](../docs/Label_Provenance_Audit_91.md) and [#93 OOT research protocol](../docs/M9_93_Inference_Safe_Benchmark.md) alongside this notebook. Saved outputs from an earlier synthetic run, if present in an existing local copy, are stale: clear outputs and rerun before interpreting anything.

## 1. Setup and safety

Use Python 3.11+ and a kernel from the environment containing `requirements.txt`. From the repository root in PowerShell:

```powershell
python -m pip install -r requirements.txt
python -m pip install jupyterlab ipykernel
python -m nltk.downloader stopwords wordnet
python -m jupyterlab notebooks\pipeline_walkthrough.ipynb
```

Select that Python kernel and Run All, or step through each cell. Run from the repository root or `notebooks` directory. First prepare the ignored local `data/processed/harmonized_marketplace_corpus.parquet` (with preserved, timezone-aware source publication times) and `data/processed/macro_lookup_table.json` using the documented ingestion pipeline; these private files are not checked in. The setup fails clearly when inputs or NLTK resources are missing, rather than substituting fixtures or fabricating dates. Optional Firebase services are never initialized.

**Training is expensive on the full local corpus** (10k+ historical eligible jobs); allow several minutes and sufficient memory. The input parquet and lookup are read-only. Research-only artifacts go into one OS temporary directory, **not `data/`, `Data/` or the repository artifact directory**. The last cell removes it; if interrupted, run that cell or restart the kernel. Re-running setup cleans its previous temporary workspace. Never save or commit notebook outputs containing local data.

In [ ]:
import importlib.util
from pathlib import Path
import sys
import tempfile

candidates = (Path.cwd(), Path.cwd().parent)
ROOT = next((p for p in candidates if p.joinpath('src', 'train_pipeline.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start the notebook from the repository root or notebooks directory.')
macro_path = ROOT / 'data/processed/macro_lookup_table.json'
parquet_path = ROOT / 'data/processed/harmonized_marketplace_corpus.parquet'
for input_path in (macro_path, parquet_path):
    if not input_path.is_file():
        raise FileNotFoundError(f'Missing ignored local input: {input_path}. Build the macro lookup and dated harmonized parquet with src.ingest_multisource first; do not substitute fixtures or synthetic timestamps.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
required = ('pandas', 'numpy', 'pyarrow', 'pycountry', 'nltk', 'sklearn', 'joblib')
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise RuntimeError(f'Missing {missing}; install requirements.txt into this notebook kernel.')

from nltk.corpus import stopwords, wordnet
try:
    stopwords.words('english')
    wordnet.ensure_loaded()
except LookupError as exc:
    raise RuntimeError('Run python -m nltk.downloader stopwords wordnet in the kernel environment, then rerun.') from exc

import numpy as np
import pandas as pd
from IPython.display import display
from src.ingest_multisource import load_macro_lookup_table
from src.train_pipeline import load_harmonized_parquet, _metric_summary, _predict_idw
from src.inference_safe_benchmark import _split_population
from src.macro_arbitrage import ContinuousMetadataNormalizer, DEFAULT_FEATURE_NAMES, fuse_coordinate_batches, fuse_coordinates
from src.nlp_pipeline import TextFeatureReducer, ensure_text_dimensions
from src.spatial_engine import DomainPartitionedKDTreeIndexer
from src.interval_synthesizer import synthesize_corridor
from src.tariff_evaluator import MpesaTariffEvaluator, DEFAULT_MPESA_TARIFF_CSV
from src.artifact_contract import file_sha256

if 'workspace' in globals():
    workspace.cleanup()
workspace = tempfile.TemporaryDirectory(prefix='price-notebook-')
WORK = Path(workspace.name)
research_dir = WORK / 'research_only'
# Declared before inspecting labels/scores; see docs/M9_93_Inference_Safe_Benchmark.md.
VALIDATION_CUTOFF = '2024-02-20T00:00:00+00:00'
TEST_CUTOFF = '2024-02-22T00:00:00+00:00'
print({'inputs': 'ignored local parquet + macro lookup', 'parquet_sha256': file_sha256(str(parquet_path)),
       'temporary_research_workspace': str(WORK), 'Firebase': 'not used'})

## 2. Inspect the local macro lookup (read-only)

This is the same ignored macro lookup used to build the local harmonized parquet: PPP is local currency per international dollar; cost-of-living is an index, not an hourly rate. Confirm the lookup matches the parquet's provenance before interpreting country factors. The historical tariff summary is **not** the current quote tariff: the quote stage reads the versioned 15-band P2P schedule from `config`.

In [ ]:
macro_records = load_macro_lookup_table(str(macro_path))
display(pd.DataFrame.from_dict(macro_records, orient='index')[
    ['country_iso2', 'ppp_lcu_per_intl_dollar', 'ppp_reference_year', 'cost_of_living_index', 'cost_index_fallback_used']
].head(8))
print({'lookup_file': macro_path.name, 'sha256': file_sha256(str(macro_path)),
       'countries': len(macro_records), 'inputs': 'ignored local lookup, read-only'})

## 3. Read the real harmonized corpus and audit source eligibility

Read the ignored local parquet produced by `src.ingest_multisource` (no network calls or writes to data directories). Its job labels are posted hourly *budgets* and the profiles' labels are asking rates, **not verified mentor payments**. `hourly_rate` is pre-arbitrage KES/hour; `harmonized_hourly_rate` is the post-factor training label. Do not show local raw descriptions, links, IDs or row-level dates in notebook output.

Ingestion filters pre-arbitrage KES/hour into [500, 35,000], but this **does not cap** the post-factor label. #93's OOT cohort then retains dated job postings only and applies a separate post-arbitrage [500, 35,000] eligibility rule. The macro inputs are historical, not live market prices.

In [ ]:
frame = load_harmonized_parquet(str(parquet_path))
required_columns = {'source_dataset', 'observation_timestamp_utc', 'harmonized_hourly_rate',
                    'mentor_country_iso2', 'client_country_iso2'}
if missing := required_columns - set(frame.columns):
    raise ValueError(f'Local corpus lacks {sorted(missing)}; rebuild it with source dates before OOT.')
display(frame.groupby('source_dataset').agg(rows=('record_id', 'size'),
                                           dated=('observation_timestamp_utc', 'count')))
print({'rows': len(frame), 'target_unit': 'KES/hour',
       'label_provenance': 'posted job budgets / profile asking rates, NOT mentor transactions'})

## 4. Predeclare OOT windows, split eligible job proxies, then fit metadata

Reuse #93's fixed chronological **train / validation / untouched test** cohort. The predeclared UTC cutoffs are above in setup, not chosen after looking at scores. `_split_population` excludes undated profiles and jobs outside post-arbitrage rate bounds; missing/naive dates among eligible jobs fail closed. It purges repeated descriptions across windows and derives frequency inputs only from past training rows. Counts and exact eligibility are reported, not silently altered.

Only training rows fit the MinMax scaler, vocabulary, SVD and peer index. Current metadata order: bilateral country factor, market saturation, industry-relative density. No target-rate column becomes a feature. This OOT test concerns **proxy posted budgets** only, not mentor-pricing accuracy or deployment eligibility.

In [ ]:
split, cohort = _split_population(frame, VALIDATION_CUTOFF, TEST_CUTOFF)
train_records = split.train.to_dict(orient='records')
normalizer = ContinuousMetadataNormalizer(str(macro_path))
train_meta = normalizer.fit_transform(train_records)
validation_meta = normalizer.transform(split.validation.to_dict(orient='records'))
test_meta = normalizer.transform(split.test.to_dict(orient='records'))
display(pd.DataFrame({'train_min': normalizer.scaler.data_min_, 'train_max': normalizer.scaler.data_max_},
                     index=DEFAULT_FEATURE_NAMES))
display(pd.DataFrame(train_meta[:5], columns=DEFAULT_FEATURE_NAMES))
print({'cohort': cohort, 'metadata_shapes': [train_meta.shape, validation_meta.shape, test_meta.shape]})

## 5. Inspect sanitization, TF-IDF and text reduction

The reducer sanitizes descriptions, fits word/word-pair TF-IDF, then fits SVD **only on historical training jobs**. Validation and test only transform through those frozen components. We show aggregate shapes, not actual local descriptions or vocabulary. `ensure_text_dimensions` enforces the 50-column text contract; SVD variance is not pricing accuracy.

In [ ]:
train_texts = split.train.raw_description.tolist()
reducer = TextFeatureReducer(n_components=50, random_state=42)
train_text = ensure_text_dimensions(reducer.fit_transform(train_texts))
validation_text = ensure_text_dimensions(reducer.transform(split.validation.raw_description.tolist()))
test_text = ensure_text_dimensions(reducer.transform(split.test.raw_description.tolist()))
print({'TFIDF_vocabulary_size': len(reducer.vectorizer.vocabulary_),
       'SVD_fitted_components': reducer.reducer.n_components,
       'text_shapes': [train_text.shape, validation_text.shape, test_text.shape],
       'explained_variance_sum_not_accuracy': reducer.explained_variance_sum()})

## 6. Fuse coordinates and retrieve peer rates

Fusion is `[50 text | bilateral factor, saturation, density]`: exactly 53 columns. The index stores **training-only** job-budget labels and IDs; validation and future test jobs are not indexed. Closer vectors have larger normalized quadratic inverse-distance weights. The API field name `verified_rate` is historical: these are **posted budgets**, not verified mentor earnings. No source descriptions or job IDs are printed.

In [ ]:
x_train = fuse_coordinate_batches(train_text, train_meta)
x_validation = fuse_coordinate_batches(validation_text, validation_meta)
x_test = fuse_coordinate_batches(test_text, test_meta)
assert x_train.shape[1] == 53
indexer = DomainPartitionedKDTreeIndexer(minimum_partition_size=5)
indexer.fit(x_train, split.train.industry_partition.tolist(),
            record_indices=split.train.record_id.tolist(), verified_rates=split.train.target_rate.tolist())
# Prefer a KE client/mentor example if the validation cohort contains one.
ke_example = split.validation.client_country_iso2.eq('KE') & split.validation.mentor_country_iso2.eq('KE')
sample_index = int(ke_example[ke_example].index[0]) if ke_example.any() else 0
sample = split.validation.iloc[sample_index]
peer_prediction = indexer.predict_base_rate(x_validation[sample_index], sample.industry_partition, k=5)
display(pd.DataFrame(peer_prediction['nearest_neighbors'])[['distance', 'verified_rate', 'idw_weight']])
print({'hybrid_shapes': [x_train.shape, x_validation.shape, x_test.shape],
       'partition_counts': indexer.partition_counts, 'requested': sample.industry_partition,
       'routed': peer_prediction['routed_partition'], 'base_KES_per_hour': peer_prediction['base_predicted_rate'],
       'peer_stddev_KES_per_hour': peer_prediction['peer_stddev'],
       'mentor_country': sample.mentor_country_iso2, 'client_country': sample.client_country_iso2})

## 7. Calculate the fee-free corridor, then add the approved-rail fee

The floor affects the lower negotiation bound, not the peer base. The configured 15-band transfer-to-M-PESA-user schedule adds one Kenyan fee to the point quote; other rails are reference-only. The final quote can exceed the corridor maximum. Unsupported Kenyan amounts fail rather than being clamped/extrapolated. This is a **research illustration on real proxy job budgets**, not a deployable price or payment. Deployment requires a separately approved tariff hash and source review.

In [ ]:
tariff_path = ROOT.joinpath(DEFAULT_MPESA_TARIFF_CSV)
tariff = MpesaTariffEvaluator.from_csv(str(tariff_path))
corridor = synthesize_corridor(peer_prediction['base_predicted_rate'], peer_prediction['peer_stddev'],
                               float(sample.bilateral_arbitrage_factor))
quote = tariff.evaluate_quote(peer_prediction['base_predicted_rate'], sample.mentor_country_iso2)
assert len(tariff.quote_bands) == 15
assert quote['final_quoted_rate'] == round(quote['base_predicted_rate'] + quote['mpesa_tariff_surcharge'], 2)
display(pd.DataFrame([{**corridor, **quote}]).drop(columns=['applied_tariff_band']))
print({'applied_band': quote['applied_tariff_band'], 'fee_free_corridor': True,
       'unit': 'KES/hour', 'not_a_verified_mentor_rate': True})

## 8. Inspect fixed OOT holdouts (proxy budgets, not a serving gate)

With the baseline configuration and predeclared chronological windows already fixed, evaluate **posted job budgets** on validation and the untouched future test. Neither holdout was used to fit the text reducer, metadata scaler, peer index or density statistics. Do not tune parameters after opening the final test. A proxy-budget score is not a verified mentor-price score or the deferred empirical gate.

No production manifest or serving artifact is published in this notebook. If you need a controlled research comparison, use the separate `src.inference_safe_benchmark` entrypoint (#93); do not silently substitute random training for OOT.

In [ ]:
validation_pred = _predict_idw(indexer, x_validation, split.validation.industry_partition.tolist(), 5)
test_pred = _predict_idw(indexer, x_test, split.test.industry_partition.tolist(), 5)
display(pd.DataFrame({
    'validation_proxy_budgets': _metric_summary(split.validation.target_rate.to_numpy(), validation_pred),
    'future_test_proxy_budgets': _metric_summary(split.test.target_rate.to_numpy(), test_pred),
}).T)
print({'protocol': 'predeclared chronological OOT, train-only fitting',
       'population': cohort['population'], 'serving_gate': 'NOT evaluated / NOT approved'})

## 9. Reload self-created temporary research artifacts and check request parity

Save **only the locally trained research components** to a temporary directory and load them back to check frozen request-time features and peer predictions. Never load untrusted downloaded joblib files. These components have **no deployable manifest**, are not approved by an OOT mentor-pricing gate, and must never be passed to `InferenceRuntime` or a protected service. No Firebase, credentials or external network is used.

In [ ]:
research_dir.mkdir(exist_ok=True)
reducer.save_artifacts(str(research_dir))
normalizer.save_artifacts(str(research_dir))
indexer.save_artifacts(str(research_dir))
restored_reducer = TextFeatureReducer.load_artifacts(str(research_dir))
restored_normalizer = ContinuousMetadataNormalizer.load_artifacts(str(research_dir), str(macro_path))
restored_indexer = DomainPartitionedKDTreeIndexer.load_artifacts(str(research_dir))
density = float(split.train.loc[split.train.industry_partition.eq(sample.industry_partition),
                               'industry_relative_density'].median())
live_meta = restored_normalizer.transform_live_metadata(
    sample.mentor_country_iso2, sample.client_country_iso2, float(sample.market_saturation_score), density)
live_vector = fuse_coordinates(ensure_text_dimensions(restored_reducer.transform([sample.raw_description])), live_meta)
reloaded_prediction = restored_indexer.predict_base_rate(live_vector, sample.industry_partition, k=5)
assert np.allclose(live_vector, x_validation[sample_index], atol=1e-5, rtol=1e-7)
assert reloaded_prediction['base_predicted_rate'] == peer_prediction['base_predicted_rate']
assert [p['peer_index'] for p in reloaded_prediction['nearest_neighbors']] == [
    p['peer_index'] for p in peer_prediction['nearest_neighbors']]
print({'request_coordinate_parity': True, 'reloaded_peer_parity': True,
       'research_components_only': sorted(p.name for p in research_dir.iterdir()),
       'deployable_manifest': False})

## 10. Deployment boundary: no API from proxy research artifacts

The research components above demonstrate training/query-time parity, **not API readiness**. `src.serve:InferenceRuntime` expects an independently trusted full artifact manifest; manufacturing one for an unvalidated proxy cohort would misleadingly appear deployable. No canonical pricing endpoint or in-memory audit is started here. For separate API contract tests use the repository's `tests/` and the local gateway documentation.

`src.deployment:app` requires Firebase verification, Firestore mentor ownership checks, backend readiness, externally pinned artifacts and a reviewed tariff pin. The notebook does not initialize Firebase or emulate protected authorization. See [deployment notes](../docs/M13.2_Container_and_CI.md) and [audit limitations](../docs/M11.2_Pricing_Audit.md).

In [ ]:
print({'gateway_started': False, 'protected_API_claim': False,
       'next_step': 'separately approve source provenance, mentor-pricing labels and external release pins'})

## 11. Cleanup and production boundaries

The production source entrypoints remain `python -m src.ingest_multisource`, `python -m src.train_pipeline`, `python -m src.inference_safe_benchmark` (research only), and the separately protected ASGI gateway. This notebook is a read-only local research runner, **not** an alternative production training/release pipeline. It must leave no data, notebook outputs, binaries or reports in Git. After cleanup, restart from setup to run again.

A real mentor-price release still needs consented verified mentor payments, an independent chronological evaluation on that population, separately approved artifact/tariff pins, and configured authorization/storage. A holdout on real **job-posting budgets** is not that evidence.

In [ ]:
workspace.cleanup()
assert not WORK.exists()
print('Temporary research artifacts removed; ignored local inputs were never modified.')